# 03. PySpark vs pandas — 같은 분석, 네 가지 방법

"채널별 완료 주문 매출과 구매자 수"를 SQL, PySpark DataFrame API, pandas API on Spark, pandas로 각각 구합니다.

In [ ]:
%%sql
SELECT u.channel, COUNT(DISTINCT o.user_id) AS buyers, SUM(o.total_amount) AS revenue
FROM orders o JOIN users u USING (user_id)
WHERE o.status = 'completed'
GROUP BY u.channel
ORDER BY revenue DESC

In [ ]:
from pyspark.sql import functions as F

orders = spark.table("orders")
users = spark.table("users")

result = (orders.filter(F.col("status") == "completed")
          .join(users, "user_id")
          .groupBy("channel")
          .agg(F.countDistinct("user_id").alias("buyers"), F.sum("total_amount").alias("revenue"))
          .orderBy(F.desc("revenue")))
display(result)

### 지연 평가(Lazy evaluation)
위 `result`는 아직 계산되지 않은 **실행 계획**입니다. `display`, `count`, `collect`, `write` 같은 **action**이 호출될 때 실행됩니다.

In [ ]:
result.explain()   # 물리 실행 계획 (BroadcastHashJoin 등을 찾아보세요)

In [ ]:
import pyspark.pandas as ps

pso = orders.pandas_api()
psu = users.pandas_api()
done = pso[pso["status"] == "completed"].merge(psu[["user_id", "channel"]], on="user_id")
done.groupby("channel").agg({"user_id": "nunique", "total_amount": "sum"}).sort_values("total_amount", ascending=False)

In [ ]:
# 작은 결과만 pandas로 가져오기 (toPandas는 드라이버 메모리에 전부 올림 → 대용량 금지)
pdf_orders = orders.filter("status = 'completed'").select("user_id", "total_amount").toPandas()
pdf_users = users.select("user_id", "channel").toPandas()
(pdf_orders.merge(pdf_users, on="user_id")
    .groupby("channel")
    .agg(buyers=("user_id", "nunique"), revenue=("total_amount", "sum"))
    .sort_values("revenue", ascending=False))

## 윈도우 함수: PySpark vs pandas

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("user_id").orderBy("order_ts")
seq = (orders.filter("status = 'completed'")
       .withColumn("order_seq", F.row_number().over(w))
       .withColumn("prev_ts", F.lag("order_ts").over(w))
       .withColumn("gap_days", F.datediff("order_ts", "prev_ts")))
display(seq.filter("order_seq > 1").select("user_id", "order_id", "order_seq", "prev_ts", "order_ts", "gap_days"), n=10)

In [ ]:
p = orders.filter("status = 'completed'").select("user_id", "order_id", "order_ts").toPandas().sort_values(["user_id", "order_ts"])
p["order_seq"] = p.groupby("user_id").cumcount() + 1
p["prev_ts"] = p.groupby("user_id")["order_ts"].shift()
p["gap_days"] = (p["order_ts"].dt.normalize() - p["prev_ts"].dt.normalize()).dt.days
p[p["order_seq"] > 1].head(10)

## pandas UDF (벡터화 UDF)
행 단위 Python UDF는 느립니다. 꼭 Python 로직이 필요하면 Arrow 기반 pandas UDF를 씁니다.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("string")
def tier(revenue: pd.Series) -> pd.Series:
    return pd.cut(revenue, bins=[-1, 100_000, 500_000, float("inf")], labels=["Light", "Regular", "VIP"]).astype(str)

rev = orders.filter("status = 'completed'").groupBy("user_id").agg(F.sum("total_amount").alias("revenue"))
display(rev.withColumn("tier", tier("revenue")).groupBy("tier").count())

## 면접 포인트
- pandas: 단일 머신·즉시 실행 / Spark: 분산·지연 실행
- 무거운 정제·집계는 Spark에서 → 작아진 결과만 `toPandas()`
- pandas 문법 그대로 분산 처리: `pyspark.pandas` (pandas API on Spark)
- UDF는 내장 함수 → pandas UDF → Python UDF 순으로 고려